# 01 — Tiền xử lý trực tiếp từ CSV

Ba client **UNSW + BoT + CIC**; 39 feature. CIC giữ đúng 1.000.000 mẫu thật (500.000 benign + 500.000 attack). Mọi mô hình dùng **CSV raw → Log → Z-score riêng từng client → clip [-8,8]**. Mean/std chỉ fit trên tập fit; đóng băng khi dùng validation/test. Split và nhãn được giữ nguyên. Kết quả mới ở `runs/main/`.

Dữ liệu được undersampling cân bằng trước khi chia train/test; row ID nguồn được lưu để kiểm tra split. Validation chia theo nhóm input trùng trong outer train. Không fit lại scaler trên validation/test và không refit trên toàn bộ train sau tuning.

In [1]:
from pathlib import Path
import os, sys, json
if os.name == 'nt':
    raise RuntimeError('Mở start_notebooks.cmd để dùng kernel WSL /opt/flids-tf.')
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p/'notebook_workflow.py').is_file() and (p/'configs/tuning.json').is_file()), None)
if PROJECT is None:
    raise FileNotFoundError('Mở notebook trong FL_IDS.')
os.chdir(PROJECT)
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))
CONFIG = PROJECT/'configs/tuning.json'
from notebook_workflow import settings, budget
C = settings(CONFIG)
RUN_ROOT = PROJECT/C['run_directory']
print('Project:', PROJECT)
print('Python:', sys.executable)
print('Run:', RUN_ROOT)


Project: /mnt/c/Users/admin/Downloads/FL_IDS
Python: /opt/flids-tf/bin/python
Run: /mnt/c/Users/admin/Downloads/FL_IDS/runs/main


In [2]:
from notebook_workflow import prepare_validation, split_table
manifest = prepare_validation(CONFIG)
display(split_table(CONFIG))
print('Data identity:', manifest['identity'])
print('Preprocessing:', manifest['policy'])

Using verified raw-CSV Log+Z-score data: /mnt/c/Users/admin/Downloads/FL_IDS/runs/main/prepared


,dataset,split,rows,benign,attack
0,unsw,fit,120369,60807,59562
1,unsw,validation,12705,5730,6975
2,unsw,test,57032,28516,28516
3,bot,fit,168102,84494,83608
4,bot,validation,20949,10031,10918
5,bot,test,81023,40512,40511
6,cic,fit,637019,317194,319825
7,cic,validation,62981,32806,30175
8,cic,test,300000,150000,150000


Data identity: b8dd47a4d13cd37749b8909e4000b13e2cccc8e4bed40874b0d6c99a51a78d8a
Preprocessing: {'method': 'per_client_log_zscore', 'log': 'sign(x) * ln(1 + abs(x)); ln(1+x) for nonnegative features', 'fit_partition': 'fit only, independently for each client and feature', 'std_ddof': 0, 'constant_std_threshold': 1e-12, 'constant_std_replacement': 1.0, 'clip': [-8.0, 8.0], 'output_dtype': 'float32', 'raw_input': 'original CSV float64 values, without MinMax reconstruction'}


In [3]:
import pandas as pd
import numpy as np
CLIENT = 'unsw'  # unsw / bot / cic
info = manifest['datasets'][CLIENT]
scaler = info['scaler']
display(pd.DataFrame({'feature': scaler['features'], 'mean_sau_log': scaler['mean'], 'std_sau_log': scaler['std']}))
records = np.memmap(RUN_ROOT/'prepared'/CLIENT/'train.bin', dtype='<f4', mode='r').reshape(-1,40)
display(pd.DataFrame(records[:5,:39], columns=scaler['features']))
print('Số mẫu fit:', scaler['n_fit'], '; nhãn 5 mẫu:', records[:5,39])

,feature,mean_sau_log,std_sau_log
0,PROTOCOL,2.326012,0.754988
1,L7_PROTO,0.620828,1.018690
2,IN_BYTES,6.798145,1.679397
3,IN_PKTS,2.480202,1.189722
4,OUT_BYTES,6.131655,3.469382
5,OUT_PKTS,2.242342,1.493754
6,TCP_FLAGS,2.235521,1.481707
7,CLIENT_TCP_FLAGS,2.228908,1.477729
8,SERVER_TCP_FLAGS,2.158244,1.466536
9,FLOW_DURATION_MILLISECONDS,1.230525,4.156868


,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,SERVER_TCP_FLAGS,FLOW_DURATION_MILLISECONDS,...,NUM_PKTS_512_TO_1024_BYTES,NUM_PKTS_1024_TO_1514_BYTES,TCP_WIN_MAX_IN,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE
0,-0.503455,-0.609438,0.776732,1.283603,1.290398,1.205493,0.740149,0.746616,0.800499,-0.296022,...,-0.404567,1.673545,0.831762,0.650830,0.579003,0.394431,-0.397326,-0.293774,-0.281453,-0.404423
1,-0.503455,1.431852,0.011070,0.071233,1.227215,0.753105,0.740149,0.746616,0.571065,-0.296022,...,-0.404567,1.476575,0.668217,0.678668,0.938350,1.164425,-0.397326,-0.293774,-0.281453,-0.404423
2,-0.503455,-0.609438,0.754696,1.252469,1.284761,1.181581,0.740149,0.746616,0.800499,-0.296022,...,-0.404567,1.629536,0.822506,0.650830,0.535984,0.303573,-0.397326,-0.293774,-0.281453,-0.404423
3,-0.503455,-0.609438,0.523554,0.620879,0.411573,0.537023,0.740149,0.746616,0.800499,-0.296022,...,-0.404567,-0.592474,0.592836,0.627076,0.899198,1.079991,-0.397326,-0.293774,-0.281453,-0.404423
4,-0.503455,-0.609438,0.974090,0.390206,0.482807,0.470022,0.740149,0.746616,0.800499,-0.296022,...,-0.404567,0.132478,0.641364,0.650830,0.737956,0.733293,-0.397326,-0.293774,-0.281453,-0.404423


Số mẫu fit: 120369 ; nhãn 5 mẫu: [0. 1. 0. 0. 0.]


Mỗi cột có mean/std riêng và mỗi client có scaler riêng. FL dùng một bộ trọng số mô hình toàn cục. Log dùng `sign(x)*ln(1+abs(x))`; với số không âm là `ln(1+x)`. Z-score dùng population std (ddof=0), cột hằng dùng scale=1, cuối cùng clip [-8,8].

Đọc trực tiếp các feature từ CSV float64. Mean/std chỉ được fit trên tập fit và dùng lại cho validation/test. Các row ID cố định trong data_splits giúp mọi mô hình và kịch bản dùng cùng dữ liệu.